## 1. Импорты и настройка путей

In [ ]:
from pathlib import Path
import sys
import time
import ast
import re
import warnings
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

from gensim import corpora, models

PROJECT_ROOT = Path(r"C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data_corpus_tex"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"
SRC_DIR = PROJECT_ROOT / "src"

for p in (OUTPUT_DIR, MODELS_DIR, FIGURES_DIR):
    p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(SRC_DIR))

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_URL = f"{OLLAMA_BASE_URL}/api/generate"
OLLAMA_MODEL = "qwen2.5:7b"

# Сколько тем брать из распределения документа для LDA и LSI.
TOP_DOC_TOPICS = 3

# Сколько слов показывать внутри каждой темы.
TOP_WORDS_PER_TOPIC = 10

# Ограничение длины тематического профиля (Cписок тем и ключевых слов)
MAX_TOPIC_PROFILE_CHARS = 6000

RANDOM_STATE = 42

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:  ", OUTPUT_DIR)
print("MODELS_DIR:  ", MODELS_DIR)

PROJECT_ROOT: C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis
OUTPUT_DIR:   C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\outputs
MODELS_DIR:   C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\models


## 2. Проверка соединения с Ollama

Перед запуском генерации нужно, чтобы Ollama была запущена локально, а модель была скачана командой `ollama pull qwen2.5:7b`.

In [ ]:
def check_ollama() -> bool:
    try:
        resp = requests.get(OLLAMA_BASE_URL, timeout=5)
        return resp.status_code == 200
    except Exception:
        return False

if check_ollama():
    print("Ollama запущена и доступна.")
else:
    print("Ollama недоступна")

Ollama запущена и доступна.


## 3. Загрузка корпуса и метаданных

In [ ]:
from text_preprocessing import read_text_file, extract_metadata


def parse_tokens(value):
    # Преобразует поле tokens к списку токенов независимо от формата сохранения
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value:
        return []
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except Exception:
            pass
    return value.split()


def resolve_tex_path(row: pd.Series) -> Path | None:
    # Находит исходный .tex-файл даже если в CSV сохранён старый абсолютный путь
    candidates = []
    if "path" in row and pd.notna(row["path"]):
        candidates.append(Path(str(row["path"])))
        candidates.append(PROJECT_ROOT / str(row["path"]))
    if "filename" in row and pd.notna(row["filename"]):
        candidates.append(DATA_DIR / str(row["filename"]))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Медленный, но надёжный резервный поиск по data_corpus_tex.
    filename = str(row.get("filename", ""))
    if filename and DATA_DIR.exists():
        matches = list(DATA_DIR.rglob(filename))
        if matches:
            return matches[0]
    return None


corpus_path = OUTPUT_DIR / "clean_corpus.csv"
if not corpus_path.exists():
    raise FileNotFoundError(
        f"Не найден {corpus_path}. Сначала нужно запустить 01_data_preprocessing.ipynb"
    )

raw_df = pd.read_csv(corpus_path)
raw_df["doc_id"] = np.arange(len(raw_df))
raw_df["tokens"] = raw_df["tokens"].apply(parse_tokens)

if "clean_text" not in raw_df.columns:
    if "cleaned_text" in raw_df.columns:
        raw_df = raw_df.rename(columns={"cleaned_text": "clean_text"})
    else:
        raw_df["clean_text"] = raw_df["tokens"].apply(lambda x: " ".join(x))

# Фильтрация нужна только для исключения пустых/почти пустых документов.
df = raw_df[
    raw_df["clean_text"].notna() & (raw_df["clean_text"].astype(str).str.len() > 100)
].copy().reset_index(drop=True)

print("Извлечение title / author / abstract из .tex файлов...")
meta_rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Метаданные"):
    tex_path = resolve_tex_path(row)
    if tex_path is None:
        meta_rows.append({"title": "", "author": "", "abstract": ""})
        continue
    try:
        raw_tex = read_text_file(tex_path)
        meta_rows.append(extract_metadata(raw_tex))
    except Exception:
        meta_rows.append({"title": "", "author": "", "abstract": ""})

meta_df = pd.DataFrame(meta_rows)

for col in ["title", "author", "abstract"]:
    if col not in df.columns:
        df[col] = meta_df[col]
    else:
        current = df[col].fillna("").astype(str)
        extracted = meta_df[col].fillna("").astype(str)
        df[col] = current.where(current.str.strip().ne(""), extracted)

# Для удобства: пустые строки вместо NaN.
for col in ["title", "author", "abstract"]:
    df[col] = df[col].fillna("").astype(str)

with_abstract_mask = df["abstract"].str.strip().str.len() > 20
without_abstract_mask = ~with_abstract_mask

df_with_abstract = df[with_abstract_mask].copy().reset_index(drop=True)
df_without_abstract = df[without_abstract_mask].copy().reset_index(drop=True)

print(f"Всего документов после фильтрации: {len(df)}")
print(f"Документов с авторским abstract:   {len(df_with_abstract)}")
print(f"Документов без abstract:           {len(df_without_abstract)}")

display(df[["filename", "title", "author", "abstract", "doc_id"]].head())

Извлечение title / author / abstract из .tex файлов...


Метаданные: 100%|██████████| 6385/6385 [00:08<00:00, 774.49it/s]

Всего документов после фильтрации: 6385
Документов с авторским abstract:   1
Документов без abstract:           6384


,filename,title,author,abstract,doc_id
0,00-5-2001.tex,Об обратимости линейных разностных операторов ...,А.Г. Баскаков,,0
1,00-5.TEX,Об обратимости линейных разностных операторов ...,А.Г. Баскаков,,1
2,00-7-2003.tex,1-строго слабо регулярные модули и кольца,А.Н. Абызов,,2
3,00-7.TEX,1-строго слабо регулярные модули и кольца,А.Н. Абызов,,3
4,01-01-2000.tex,Оценка числа решений однородной задачи Римана ...,И.А. Бикчантаев,,4


## 4. Загрузка результатов тематического моделирования

In [5]:
model_status = {}

# Gensim dictionary / corpus, общие для LDA и LSI.
dictionary = None
corpus_bow = None
try:
    dictionary = corpora.Dictionary.load(str(MODELS_DIR / "gensim_dictionary.dict"))
    corpus_bow = list(corpora.MmCorpus(str(MODELS_DIR / "gensim_corpus.mm")))
    model_status["gensim_corpus"] = f"загружен корпус: {len(corpus_bow)} документов, словарь: {len(dictionary)} терминов"
except Exception as e:
    model_status["gensim_corpus"] = f"не загружен: {e}"

# LDA.
lda_model = None
lda_path = MODELS_DIR / "lda_model_best.model"
if lda_path.exists():
    try:
        try:
            lda_model = models.LdaModel.load(str(lda_path))
        except Exception:
            lda_model = models.LdaMulticore.load(str(lda_path))
        model_status["LDA"] = f"загружена модель: {lda_model.num_topics} тем"
    except Exception as e:
        model_status["LDA"] = f"ошибка загрузки: {e}"
else:
    model_status["LDA"] = "файл модели не найден"

# LSI + TF-IDF.
lsi_model = None
tfidf_model = None
lsi_path = MODELS_DIR / "lsi_model_best.model"
tfidf_path = MODELS_DIR / "tfidf_model.model"
if lsi_path.exists() and corpus_bow is not None:
    try:
        lsi_model = models.LsiModel.load(str(lsi_path))
        if tfidf_path.exists():
            tfidf_model = models.TfidfModel.load(str(tfidf_path))
        else:
            tfidf_model = models.TfidfModel(corpus_bow)
        model_status["LSI"] = f"загружена модель: {lsi_model.num_topics} тем"
    except Exception as e:
        model_status["LSI"] = f"ошибка загрузки: {e}"
else:
    model_status["LSI"] = "файл модели не найден или не загружен gensim-корпус"

# BERTopic: CSV с темами документов + сама модель для ключевых слов тем, если библиотека доступна.
bertopic_docs = None
bertopic_model = None
bertopic_docs_path = OUTPUT_DIR / "documents_with_topics.csv"
if bertopic_docs_path.exists():
    try:
        bertopic_docs = pd.read_csv(bertopic_docs_path)
        model_status["BERTopic documents"] = f"загружено строк: {len(bertopic_docs)}"
    except Exception as e:
        model_status["BERTopic documents"] = f"ошибка загрузки CSV: {e}"
else:
    model_status["BERTopic documents"] = "CSV не найден"

bertopic_model_path = MODELS_DIR / "bertopic_model"
if bertopic_model_path.exists():
    try:
        from bertopic import BERTopic
        bertopic_model = BERTopic.load(str(bertopic_model_path))
        model_status["BERTopic model"] = "загружена модель для слов тем"
    except Exception as e:
        model_status["BERTopic model"] = f"модель не загружена, будут использоваться только номера тем: {e}"
else:
    model_status["BERTopic model"] = "папка модели не найдена"

# BigARTM: чаще всего используем сохранённую таблицу с темами документов.
bigartm_docs = None
bigartm_docs_path = OUTPUT_DIR / "bigartm_documents_topics.csv"
if bigartm_docs_path.exists():
    try:
        bigartm_docs = pd.read_csv(bigartm_docs_path)
        model_status["BigARTM documents"] = f"загружено строк: {len(bigartm_docs)}"
    except Exception as e:
        model_status["BigARTM documents"] = f"ошибка загрузки CSV: {e}"
else:
    model_status["BigARTM documents"] = "CSV не найден"

# Необязательный файл с ключевыми словами BigARTM-тем, если он был сохранён отдельно.
bigartm_topic_words = {}
bigartm_words_path = OUTPUT_DIR / "bigartm_topic_words.csv"
if bigartm_words_path.exists():
    try:
        tmp = pd.read_csv(bigartm_words_path)
        # Поддерживаются два формата: topic/words или topic_name/word/weight.
        if {"topic", "words"}.issubset(tmp.columns):
            for _, r in tmp.iterrows():
                words = [w.strip() for w in str(r["words"]).split(",") if w.strip()]
                bigartm_topic_words[str(r["topic"])] = words
        elif {"topic_name", "word"}.issubset(tmp.columns):
            for topic_name, part in tmp.groupby("topic_name"):
                bigartm_topic_words[str(topic_name)] = part["word"].astype(str).tolist()
        model_status["BigARTM topic words"] = f"загружено тем: {len(bigartm_topic_words)}"
    except Exception as e:
        model_status["BigARTM topic words"] = f"ошибка загрузки: {e}"
else:
    model_status["BigARTM topic words"] = "файл не найден, будут использоваться номера/веса тем"

print("Статус загрузки моделей и таблиц:")
for k, v in model_status.items():
    print(f"  {k}: {v}")

Статус загрузки моделей и таблиц:
  gensim_corpus: загружен корпус: 6385 документов, словарь: 10000 терминов
  LDA: загружена модель: 25 тем
  LSI: загружена модель: 5 тем
  BERTopic documents: загружено строк: 6385
  BERTopic model: загружена модель для слов тем
  BigARTM documents: загружено строк: 6385
  BigARTM topic words: файл не найден, будут использоваться номера/веса тем


## 5. Формирование тематического профиля документа

In [ ]:
def _join_words(words, max_words: int = TOP_WORDS_PER_TOPIC) -> str:
    clean_words = []
    for w in words[:max_words]:
        w = str(w).strip()
        if w and w not in clean_words:
            clean_words.append(w)
    return ", ".join(clean_words)


def _format_weight(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    try:
        return f"; вес={float(value):.3f}"
    except Exception:
        return ""


def get_lda_topics_for_doc(doc_id: int, top_n: int = TOP_DOC_TOPICS) -> list[dict]:
    if lda_model is None or corpus_bow is None or doc_id >= len(corpus_bow):
        return []
    topic_dist = lda_model.get_document_topics(corpus_bow[doc_id], minimum_probability=0)
    topic_dist = sorted(topic_dist, key=lambda x: x[1], reverse=True)[:top_n]
    result = []
    for topic_id, weight in topic_dist:
        words = [w for w, _ in lda_model.show_topic(topic_id, topn=TOP_WORDS_PER_TOPIC)]
        result.append({
            "source": "LDA",
            "topic_id": f"Тема {topic_id + 1}",
            "weight": float(weight),
            "words": words,
        })
    return result


def get_lsi_topics_for_doc(doc_id: int, top_n: int = TOP_DOC_TOPICS) -> list[dict]:
    if lsi_model is None or tfidf_model is None or corpus_bow is None or doc_id >= len(corpus_bow):
        return []
    lsi_dist = list(lsi_model[tfidf_model[corpus_bow[doc_id]]])
    lsi_dist = sorted(lsi_dist, key=lambda x: abs(x[1]), reverse=True)[:top_n]
    result = []
    for topic_id, weight in lsi_dist:
        # Для LSI знак веса может быть отрицательным, поэтому в профиль добавляется исходный вес,
        # Но для ключевых слов берутся термины с наибольшим вкладом по модулю.
        raw_topic = lsi_model.show_topic(topic_id, topn=max(20, TOP_WORDS_PER_TOPIC))
        raw_topic = sorted(raw_topic, key=lambda x: abs(x[1]), reverse=True)
        words = [w for w, _ in raw_topic[:TOP_WORDS_PER_TOPIC]]
        result.append({
            "source": "LSI",
            "topic_id": f"Тема {topic_id + 1}",
            "weight": float(weight),
            "words": words,
        })
    return result


# Быстрые индексы по filename.
bertopic_by_filename = {}
if bertopic_docs is not None and "filename" in bertopic_docs.columns:
    bertopic_by_filename = bertopic_docs.drop_duplicates("filename").set_index("filename").to_dict("index")

bigartm_by_filename = {}
if bigartm_docs is not None and "filename" in bigartm_docs.columns:
    bigartm_by_filename = bigartm_docs.drop_duplicates("filename").set_index("filename").to_dict("index")


def get_bertopic_for_doc(filename: str) -> list[dict]:
    if not bertopic_by_filename:
        return []
    row = bertopic_by_filename.get(filename)
    if not row:
        return []

    topic_id = row.get("topic")
    if topic_id is None or pd.isna(topic_id):
        return []
    try:
        topic_id_int = int(topic_id)
    except Exception:
        topic_id_int = topic_id

    words = []
    if bertopic_model is not None and topic_id_int != -1:
        try:
            words = [w for w, _ in bertopic_model.get_topic(topic_id_int)[:TOP_WORDS_PER_TOPIC]]
        except Exception:
            words = []

    label = "шум / выброс" if topic_id_int == -1 else f"Тема {topic_id_int}"
    return [{
        "source": "BERTopic",
        "topic_id": label,
        "weight": None,
        "words": words,
    }]


def _bigartm_words_for_topic(topic_label: str) -> list[str]:
    candidates = [topic_label]
    # В 05_bigartm dominant_topic часто выглядит как topic_0, а веса сохранены как topic_1_weight.
    m = re.search(r"topic_(\d+)", str(topic_label))
    if m:
        zero_based = int(m.group(1))
        candidates.extend([f"topic_{zero_based}", f"topic_{zero_based + 1}", f"topic_{zero_based + 1}_weight"])
    for c in candidates:
        if str(c) in bigartm_topic_words:
            return bigartm_topic_words[str(c)][:TOP_WORDS_PER_TOPIC]
    return []


def get_bigartm_for_doc(filename: str, top_n: int = TOP_DOC_TOPICS) -> list[dict]:
    if not bigartm_by_filename:
        return []
    row = bigartm_by_filename.get(filename)
    if not row:
        return []

    result = []
    dominant = row.get("dominant_topic", "")
    if pd.notna(dominant) and str(dominant).strip():
        result.append({
            "source": "BigARTM",
            "topic_id": f"доминирующая {dominant}",
            "weight": None,
            "words": _bigartm_words_for_topic(str(dominant)),
        })

    # Добавляется топ тем по весам, если в CSV есть topic_*_weight.
    weight_items = []
    for col, val in row.items():
        if re.fullmatch(r"topic_\d+_weight", str(col)):
            try:
                weight_items.append((col, float(val)))
            except Exception:
                pass
    weight_items = sorted(weight_items, key=lambda x: x[1], reverse=True)[:top_n]

    for col, weight in weight_items:
        result.append({
            "source": "BigARTM",
            "topic_id": col.replace("_weight", ""),
            "weight": weight,
            "words": _bigartm_words_for_topic(col.replace("_weight", "")),
        })

    # Удаление возможных дубликатов
    seen = set()
    unique = []
    for item in result:
        key = (item["source"], item["topic_id"])
        if key not in seen:
            unique.append(item)
            seen.add(key)
    return unique


def get_fallback_document_keywords(tokens: list[str], n: int = 20) -> list[str]:
    # Резервные ключевые слова документа, если тематические модели недоступны
    if not tokens:
        return []
    counts = pd.Series(tokens).value_counts()
    return counts.head(n).index.astype(str).tolist()


def build_topic_items(row: pd.Series) -> list[dict]:
    doc_id = int(row["doc_id"])
    filename = str(row["filename"])
    items = []
    items.extend(get_lda_topics_for_doc(doc_id))
    items.extend(get_lsi_topics_for_doc(doc_id))
    items.extend(get_bertopic_for_doc(filename))
    items.extend(get_bigartm_for_doc(filename))
    return items


def format_topic_items_for_display(items: list[dict]) -> str:
    # Формирует компактный текст со списком тем для просмотра рядом с аннотацией
    lines = []
    for item in items:
        words_text = _join_words(item.get("words", []))
        line = f"{item['source']}: {item['topic_id']}{_format_weight(item.get('weight'))}"
        if words_text:
            line += f"; ключевые слова: {words_text}"
        lines.append(line)
    return "\n".join(lines)


def build_document_topics_text(row: pd.Series) -> str:
    # Возвращает темы, относящиеся к конкретному документу, для вывода и CSV
    items = build_topic_items(row)
    if items:
        return format_topic_items_for_display(items)

    # Защита от ситуации, когда тематические модели ещё не обучены или не сопоставились с файлом.
    keywords = get_fallback_document_keywords(row.get("tokens", []), n=20)
    if keywords:
        return "Тематические модели не найдены или не сопоставлены. Резервные ключевые слова: " + _join_words(keywords, max_words=20)
    return "Темы не найдены."


def build_topic_profile(row: pd.Series) -> str:
    # Создаёт текстовый профиль из тем. Именно его получает LLM
    title = str(row.get("title", "")).strip()
    author = str(row.get("author", "")).strip()
    filename = str(row.get("filename", "")).strip()

    lines = []
    lines.append(f"Файл: {filename}")
    if title:
        lines.append(f"Название статьи: {title}")
    if author:
        lines.append(f"Авторы: {author}")

    lines.append("")
    lines.append("Тематический профиль документа, полученный методами тематического моделирования:")

    items = build_topic_items(row)
    if items:
        for item in items:
            words_text = _join_words(item.get("words", []))
            if words_text:
                lines.append(
                    f"- {item['source']}: {item['topic_id']}{_format_weight(item.get('weight'))}; "
                    f"ключевые слова темы: {words_text}."
                )
            else:
                lines.append(
                    f"- {item['source']}: {item['topic_id']}{_format_weight(item.get('weight'))}."
                )
    else:
        # Это не основной сценарий, а защита от ситуации, когда модели ещё не обучены.
        keywords = get_fallback_document_keywords(row.get("tokens", []), n=20)
        lines.append(
            "- Тематические модели не найдены или не сопоставлены с документом. "
            "Резервные ключевые слова документа: " + _join_words(keywords, max_words=20) + "."
        )

    lines.append("")
    lines.append(
        "Сформируй аннотацию осторожно: не добавляй точные теоремы, методы и результаты, "
        "если они не следуют из тематического профиля."
    )

    profile = "\n".join(lines)
    return profile[:MAX_TOPIC_PROFILE_CHARS]


# Проверка на первом документе.
example_profile = build_topic_profile(df.iloc[0])
example_topics_text = build_document_topics_text(df.iloc[0])
print(example_profile[:2500])
print("\nТемы для отображения рядом с аннотацией:")
print(example_topics_text[:2500])


Файл: 00-5-2001.tex
Название статьи: Об обратимости линейных разностных операторов с постоянными коэффициентами
Авторы: А.Г. Баскаков

Тематический профиль документа, полученный методами тематического моделирования:
- LDA: Тема 9; вес=0.854; ключевые слова темы: оператор, функция, пусть, неравенство, пространство, любой, следовать, множество, сила, существовать.
- LDA: Тема 6; вес=0.071; ключевые слова темы: группа, алгебра, кольцо, модуль, поле, элемент, любой, подгруппа, наз, класс.
- LDA: Тема 24; вес=0.041; ключевые слова темы: уравнение, функция, система, дифференциальный, удовлетворять, краевой, коэффициент, порядок, область, оператор.
- LSI: Тема 4; вес=0.335; ключевые слова темы: оператор, распределение, случайный, вероятность, частица, процесс, модель, неравенство, состояние, квантовый.
- LSI: Тема 1; вес=0.281; ключевые слова темы: оператор, уравнение, группа, пространство, поле, функция, матрица, наз, система, неравенство.
- LSI: Тема 3; вес=0.157; ключевые слова темы: групп

## 6. Генерация аннотации по тематическому профилю

Главное отличие от предыдущего модуля: функция `generate_summary_from_topics` принимает `topic_profile`, а не `clean_text`. Prompt явно сообщает модели, что на входе находится тематический профиль, а не полный текст статьи.

In [ ]:
PROMPT_TEMPLATE_TOPICS = """Ты — научный редактор, специализирующийся на математических публикациях.
На входе дан не полный текст статьи, а тематический профиль документа, полученный методами LDA, LSI, BERTopic и BigARTM.

По этому тематическому профилю напиши краткую аннотацию статьи на русском языке: 3–5 предложений.
Аннотация должна отражать:
1) основную математическую область и предполагаемую задачу;
2) ключевые объекты, методы или понятия, если они видны из тем;
3) общий характер результата без выдумывания конкретных утверждений.

Не пересказывай список тем механически. Не упоминай названия моделей LDA, LSI, BERTopic и BigARTM в самой аннотации.
Если по темам нельзя уверенно определить метод или результат, используй осторожные формулировки: «работа относится к...», «рассматриваются...», «основное внимание уделяется...».
Пиши только аннотацию, без пояснений и вводных фраз.

Тематический профиль:
{topic_profile}

Аннотация:"""


def build_prompt(topic_profile: str) -> str:
    return PROMPT_TEMPLATE_TOPICS.format(topic_profile=str(topic_profile)[:MAX_TOPIC_PROFILE_CHARS])


def generate_summary_from_topics(
    topic_profile: str,
    model: str = OLLAMA_MODEL,
    timeout: int = 120,
) -> str:
    # Отправляет в Ollama только тематический профиль документа и возвращает аннотацию
    prompt = build_prompt(topic_profile)
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.25,
            "top_p": 0.9,
            "num_predict": 300,
        },
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except requests.exceptions.Timeout:
        return "[ОШИБКА: таймаут]"
    except Exception as e:
        return f"[ОШИБКА: {e}]"


# Самопроверка: в prompt не используется полный текст статьи.
test_prompt = build_prompt(example_profile)
assert "Текст статьи:" not in test_prompt
assert "clean_text" not in test_prompt
print("Функции генерации готовы. В prompt передаётся тематический профиль, а не текст статьи.")

Функции генерации готовы. В prompt передаётся тематический профиль, а не текст статьи.


## 7. Эксперимент 1: один файл с авторским abstract

In [ ]:
SELECTED_FILENAME_WITH_ABSTRACT = None  # например: "some_file.tex", если нужно выбрать файл вручную

if df_with_abstract.empty:
    raise ValueError("В корпусе не найдено документов с авторским abstract.")

if SELECTED_FILENAME_WITH_ABSTRACT:
    selected_df = df_with_abstract[df_with_abstract["filename"] == SELECTED_FILENAME_WITH_ABSTRACT].copy()
    if selected_df.empty:
        raise ValueError(f"Файл с abstract не найден: {SELECTED_FILENAME_WITH_ABSTRACT}")
else:
    selected_df = df_with_abstract.head(1).copy()

one_row = selected_df.iloc[0]
one_profile = build_topic_profile(one_row)

print("Выбран документ с авторским abstract:")
print("Файл:", one_row["filename"])
print("Заголовок:", one_row.get("title", ""))
print("\nВход для LLM — тематический профиль:")
print("-" * 80)
print(one_profile)
print("-" * 80)

Выбран документ с авторским abstract:
Файл: polieces.tex
Заголовок: 

Вход для LLM — тематический профиль:
--------------------------------------------------------------------------------
Файл: polieces.tex

Тематический профиль документа, полученный методами тематического моделирования:
- LDA: Тема 7; вес=0.884; ключевые слова темы: система, информационный, информация, данных, технология, научный, процесс, исследование, область, средство.
- LDA: Тема 17; вес=0.091; ключевые слова темы: объект, текст, модель, слово, данных, признак, язык, анализ, поиск, класс.
- LDA: Тема 1; вес=0.013; ключевые слова темы: движение, система, тело, время, скорость, уравнение, модель, сила, координата, точка.
- LSI: Тема 2; вес=-0.133; ключевые слова темы: казань, редакция, отпечатать, редактор, факс, макет, цена, оригинал, тело, университетский.
- LSI: Тема 4; вес=-0.077; ключевые слова темы: оператор, распределение, случайный, вероятность, частица, процесс, модель, неравенство, состояние, квантовый.
- 

In [9]:
start_time = time.time()
one_topics_text = build_document_topics_text(one_row)
one_summary = generate_summary_from_topics(one_profile)
elapsed = time.time() - start_time

one_result = pd.DataFrame([{
    "filename": one_row["filename"],
    "title": one_row.get("title", ""),
    "author": one_row.get("author", ""),
    "abstract": one_row.get("abstract", ""),
    "document_topics": one_topics_text,
    "topic_profile": one_profile,
    "generated_summary": one_summary,
    "generation_seconds": round(elapsed, 2),
    "input_type": "topics_only",
}])

one_output_path = OUTPUT_DIR / "summary_from_topics_with_abstract_one.csv"
one_result.to_csv(one_output_path, index=False, encoding="utf-8")

print(f"Генерация завершена за {elapsed:.1f} сек.")
print("\nТемы документа:")
print(one_topics_text)
print("\nАвторский abstract:")
print(str(one_row.get("abstract", ""))[:1200])
print("\nСгенерированная аннотация по темам:")
print(one_summary)
print("\nСохранено:", one_output_path)

display(one_result[["filename", "title", "document_topics", "generated_summary", "generation_seconds", "input_type"]])


Генерация завершена за 49.2 сек.

Темы документа:
LDA: Тема 7; вес=0.884; ключевые слова: система, информационный, информация, данных, технология, научный, процесс, исследование, область, средство
LDA: Тема 17; вес=0.091; ключевые слова: объект, текст, модель, слово, данных, признак, язык, анализ, поиск, класс
LDA: Тема 1; вес=0.013; ключевые слова: движение, система, тело, время, скорость, уравнение, модель, сила, координата, точка
LSI: Тема 2; вес=-0.133; ключевые слова: казань, редакция, отпечатать, редактор, факс, макет, цена, оригинал, тело, университетский
LSI: Тема 4; вес=-0.077; ключевые слова: оператор, распределение, случайный, вероятность, частица, процесс, модель, неравенство, состояние, квантовый
LSI: Тема 1; вес=0.077; ключевые слова: оператор, уравнение, группа, пространство, поле, функция, матрица, наз, система, неравенство
BERTopic: шум / выброс
BigARTM: доминирующая topic_24
BigARTM: topic_25; вес=0.839
BigARTM: topic_4; вес=0.100
BigARTM: topic_9; вес=0.044

Авторски

,filename,title,document_topics,generated_summary,generation_seconds,input_type
0,polieces.tex,,LDA: Тема 7; вес=0.884; ключевые слова: систем...,Работа относится к области информационных техн...,49.19,topics_only


## 8. Оценка для файла с авторским abstract

In [10]:
metrics = {"filename": one_row["filename"]}
reference = str(one_row.get("abstract", "")).strip()
hypothesis = str(one_summary).strip()

# ROUGE.
try:
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=False,
    )
    rouge_scores = scorer.score(reference, hypothesis)
    for name, score in rouge_scores.items():
        metrics[f"{name}_p"] = score.precision
        metrics[f"{name}_r"] = score.recall
        metrics[f"{name}_f"] = score.fmeasure

    print("ROUGE:")
    print(f"  ROUGE-1 F1: {metrics['rouge1_f']:.4f}")
    print(f"  ROUGE-2 F1: {metrics['rouge2_f']:.4f}")
    print(f"  ROUGE-L F1: {metrics['rougeL_f']:.4f}")
except Exception as e:
    print("ROUGE не рассчитан:", e)

# BERTScore можно отключить, если нет нужной библиотеки или модель долго загружается.
RUN_BERTSCORE = True
if RUN_BERTSCORE:
    try:
        from bert_score import score as bert_score

        P, R, F1 = bert_score(
            [hypothesis],
            [reference],
            lang="ru",
            model_type="intfloat/multilingual-e5-small",
            num_layers=12,
            verbose=True,
        )
        metrics["bertscore_p"] = float(P[0])
        metrics["bertscore_r"] = float(R[0])
        metrics["bertscore_f1"] = float(F1[0])

        print("\nBERTScore:")
        print(f"  Precision: {metrics['bertscore_p']:.4f}")
        print(f"  Recall:    {metrics['bertscore_r']:.4f}")
        print(f"  F1:        {metrics['bertscore_f1']:.4f}")
    except Exception as e:
        print("BERTScore не рассчитан:", e)

metrics_df = pd.DataFrame([metrics])
metrics_output_path = OUTPUT_DIR / "summary_from_topics_with_abstract_metrics.csv"
metrics_df.to_csv(metrics_output_path, index=False, encoding="utf-8")
print("\nМетрики сохранены:", metrics_output_path)

display(metrics_df)

ROUGE:
  ROUGE-1 F1: 0.0000
  ROUGE-2 F1: 0.0000
  ROUGE-L F1: 0.0000
calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  2.51it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 63.96it/s]

done in 0.42 seconds, 2.39 sentences/sec

BERTScore:
  Precision: 0.7814
  Recall:    0.7272
  F1:        0.7533

Метрики сохранены: C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\outputs\summary_from_topics_with_abstract_metrics.csv


,filename,rouge1_p,rouge1_r,rouge1_f,rouge2_p,rouge2_r,rouge2_f,rougeL_p,rougeL_r,rougeL_f,bertscore_p,bertscore_r,bertscore_f1
0,polieces.tex,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0.78141,0.727206,0.753334


## 9. Эксперимент 2: 10 файлов без авторского abstract

In [ ]:
NO_ABSTRACT_SAMPLE_SIZE = 10

# Можно задать список файлов. Если список пустой, берутся первые 10 документов без abstract.
SELECTED_FILENAMES_WITHOUT_ABSTRACT = []

if SELECTED_FILENAMES_WITHOUT_ABSTRACT:
    target_without_abs = df_without_abstract[
        df_without_abstract["filename"].isin(SELECTED_FILENAMES_WITHOUT_ABSTRACT)
    ].copy()
    missing = set(SELECTED_FILENAMES_WITHOUT_ABSTRACT) - set(target_without_abs["filename"])
    if missing:
        print("Не найдены указанные файлы:", sorted(missing))
else:
    target_without_abs = df_without_abstract.head(NO_ABSTRACT_SAMPLE_SIZE).copy()

target_without_abs = target_without_abs.reset_index(drop=True)

if len(target_without_abs) == 0:
    raise ValueError("Не найдено документов без abstract для генерации.")

print(f"Документов без abstract для генерации: {len(target_without_abs)}")
display(target_without_abs[["filename", "title", "author", "doc_id"]])

Документов без abstract для генерации: 10


,filename,title,author,doc_id
0,00-5-2001.tex,Об обратимости линейных разностных операторов ...,А.Г. Баскаков,0
1,00-5.TEX,Об обратимости линейных разностных операторов ...,А.Г. Баскаков,1
2,00-7-2003.tex,1-строго слабо регулярные модули и кольца,А.Н. Абызов,2
3,00-7.TEX,1-строго слабо регулярные модули и кольца,А.Н. Абызов,3
4,01-01-2000.tex,Оценка числа решений однородной задачи Римана ...,И.А. Бикчантаев,4
5,01-01-2005.tex,Об одном аддитивном методе для нестационарных ...,", Н.Г. Жадаева",5
6,01-01.TEX,Об одном аддитивном методе для нестационарных ...,", Н.Г. Жадаева",6
7,01-02-2000.tex,Устойчивость решений нелинейных почти периодич...,Н.В. Алексенко,7
8,01-02.TEX,Устойчивость решений нелинейных почти периодич...,Н.В. Алексенко,8
9,01-03-2000.tex,О слабом законе больших чисел в банаховых прос...,"О.В. Антонова, С.А. Бронникова, В.В. Давыдова,...",9


In [12]:
records = []
start_time = time.time()

for _, row in tqdm(target_without_abs.iterrows(), total=len(target_without_abs), desc="Генерация по темам"):
    topic_profile = build_topic_profile(row)
    document_topics = build_document_topics_text(row)
    summary = generate_summary_from_topics(topic_profile)
    records.append({
        "filename": row["filename"],
        "title": row.get("title", ""),
        "author": row.get("author", ""),
        "document_topics": document_topics,
        "topic_profile": topic_profile,
        "generated_summary": summary,
        "input_type": "topics_only",
    })

elapsed = time.time() - start_time
without_abs_results = pd.DataFrame(records)

without_abs_output_path = OUTPUT_DIR / "summaries_from_topics_without_abstract_10.csv"
without_abs_results.to_csv(without_abs_output_path, index=False, encoding="utf-8")

print(f"Готово: {len(without_abs_results)} документов за {elapsed:.1f} сек.")
print("Сохранено:", without_abs_output_path)

display(without_abs_results[["filename", "title", "document_topics", "generated_summary", "input_type"]])


Генерация по темам: 100%|██████████| 10/10 [05:39<00:00, 33.92s/it]

Готово: 10 документов за 339.2 сек.
Сохранено: C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\outputs\summaries_from_topics_without_abstract_10.csv


,filename,title,document_topics,generated_summary,input_type
0,00-5-2001.tex,Об обратимости линейных разностных операторов ...,LDA: Тема 9; вес=0.854; ключевые слова: операт...,Работа относится к области линейной алгебры и ...,topics_only
1,00-5.TEX,Об обратимости линейных разностных операторов ...,LDA: Тема 9; вес=0.854; ключевые слова: операт...,Работа относится к области математического ана...,topics_only
2,00-7-2003.tex,1-строго слабо регулярные модули и кольца,LDA: Тема 6; вес=0.735; ключевые слова: группа...,Работа относится к области алгебры и теории ко...,topics_only
3,00-7.TEX,1-строго слабо регулярные модули и кольца,LDA: Тема 6; вес=0.735; ключевые слова: группа...,"Работа относится к алгебре и теории колец, с а...",topics_only
4,01-01-2000.tex,Оценка числа решений однородной задачи Римана ...,LDA: Тема 13; вес=0.605; ключевые слова: точка...,Работа относится к области математического ана...,topics_only
5,01-01-2005.tex,Об одном аддитивном методе для нестационарных ...,LDA: Тема 24; вес=0.268; ключевые слова: уравн...,Работа относится к области математического мод...,topics_only
6,01-01.TEX,Об одном аддитивном методе для нестационарных ...,LDA: Тема 24; вес=0.268; ключевые слова: уравн...,Работа относится к численному решению нестацио...,topics_only
7,01-02-2000.tex,Устойчивость решений нелинейных почти периодич...,LDA: Тема 9; вес=0.504; ключевые слова: операт...,Работа относится к исследованию устойчивости р...,topics_only
8,01-02.TEX,Устойчивость решений нелинейных почти периодич...,LDA: Тема 9; вес=0.504; ключевые слова: операт...,Работа относится к области математического ана...,topics_only
9,01-03-2000.tex,О слабом законе больших чисел в банаховых прос...,LDA: Тема 10; вес=0.446; ключевые слова: распр...,Работа относится к теории вероятностей и матем...,topics_only


## 10. Просмотр итоговых аннотаций и сохранение общей таблицы


In [13]:
combined_df = pd.concat([
    one_result.assign(has_author_abstract=True),
    without_abs_results.assign(abstract="", generation_seconds=np.nan, has_author_abstract=False),
], ignore_index=True, sort=False)

combined_output_path = OUTPUT_DIR / "summaries_from_topics_selected_11.csv"
combined_df.to_csv(combined_output_path, index=False, encoding="utf-8")

print("Общий файл сохранён:", combined_output_path)
print(f"Всего строк: {len(combined_df)}")

preview_cols = [
    "filename",
    "title",
    "document_topics",
    "generated_summary",
    "has_author_abstract",
]
display(combined_df[preview_cols])

for _, row in combined_df.iterrows():
    print("\n" + "=" * 90)
    print("Файл:", row["filename"])
    if str(row.get("title", "")).strip():
        print("Название:", row.get("title", ""))
    print("Есть авторский abstract:", bool(row.get("has_author_abstract", False)))
    print("\nТемы документа:")
    topics_text = str(row.get("document_topics", "")).strip()
    print(topics_text if topics_text else "Темы не найдены.")
    print("\nАннотация:")
    print(row["generated_summary"])


Общий файл сохранён: C:\Users\anmrt\Desktop\Useful shit\Диплом\graduation_thesis\outputs\summaries_from_topics_selected_11.csv
Всего строк: 11


,filename,title,document_topics,generated_summary,has_author_abstract
0,polieces.tex,,LDA: Тема 7; вес=0.884; ключевые слова: систем...,Работа относится к области информационных техн...,True
1,00-5-2001.tex,Об обратимости линейных разностных операторов ...,LDA: Тема 9; вес=0.854; ключевые слова: операт...,Работа относится к области линейной алгебры и ...,False
2,00-5.TEX,Об обратимости линейных разностных операторов ...,LDA: Тема 9; вес=0.854; ключевые слова: операт...,Работа относится к области математического ана...,False
3,00-7-2003.tex,1-строго слабо регулярные модули и кольца,LDA: Тема 6; вес=0.735; ключевые слова: группа...,Работа относится к области алгебры и теории ко...,False
4,00-7.TEX,1-строго слабо регулярные модули и кольца,LDA: Тема 6; вес=0.735; ключевые слова: группа...,"Работа относится к алгебре и теории колец, с а...",False
5,01-01-2000.tex,Оценка числа решений однородной задачи Римана ...,LDA: Тема 13; вес=0.605; ключевые слова: точка...,Работа относится к области математического ана...,False
6,01-01-2005.tex,Об одном аддитивном методе для нестационарных ...,LDA: Тема 24; вес=0.268; ключевые слова: уравн...,Работа относится к области математического мод...,False
7,01-01.TEX,Об одном аддитивном методе для нестационарных ...,LDA: Тема 24; вес=0.268; ключевые слова: уравн...,Работа относится к численному решению нестацио...,False
8,01-02-2000.tex,Устойчивость решений нелинейных почти периодич...,LDA: Тема 9; вес=0.504; ключевые слова: операт...,Работа относится к исследованию устойчивости р...,False
9,01-02.TEX,Устойчивость решений нелинейных почти периодич...,LDA: Тема 9; вес=0.504; ключевые слова: операт...,Работа относится к области математического ана...,False



Файл: polieces.tex
Есть авторский abstract: True

Темы документа:
LDA: Тема 7; вес=0.884; ключевые слова: система, информационный, информация, данных, технология, научный, процесс, исследование, область, средство
LDA: Тема 17; вес=0.091; ключевые слова: объект, текст, модель, слово, данных, признак, язык, анализ, поиск, класс
LDA: Тема 1; вес=0.013; ключевые слова: движение, система, тело, время, скорость, уравнение, модель, сила, координата, точка
LSI: Тема 2; вес=-0.133; ключевые слова: казань, редакция, отпечатать, редактор, факс, макет, цена, оригинал, тело, университетский
LSI: Тема 4; вес=-0.077; ключевые слова: оператор, распределение, случайный, вероятность, частица, процесс, модель, неравенство, состояние, квантовый
LSI: Тема 1; вес=0.077; ключевые слова: оператор, уравнение, группа, пространство, поле, функция, матрица, наз, система, неравенство
BERTopic: шум / выброс
BigARTM: доминирующая topic_24
BigARTM: topic_25; вес=0.839
BigARTM: topic_4; вес=0.100
BigARTM: topic_9; ве